# Region 6 manual trace definition for Mayassi-style unrolling

This notebook makes the previously implicit manual annotation step explicit. It must be reviewed before `B3_Region6_Mayassi_unrolling_method_audit.ipynb` is run.

**Input population:** all cells in `Region_6_spatial_passQC.rds`, which is the returned object restricted to `primary_include_revised == TRUE`.

**Smooth-muscle guide:** `RefAll_subtype_predicted.id == "Smooth muscle"`. `Xenium_cluster_subtype` is not used.


## What was manual and what is reproducible

The 39 control points were manually entered during the Region 6 feasibility review by visually following the outer band of cells transferred as Mayassi-reference `Smooth muscle`, from the outer free edge through successive turns to the inner hook. Coordinates were rounded to practical plotting positions; they were not inferred by an optimization algorithm and there is no hidden click log.

Exact computational replication is achieved by version-controlling those coordinates. Independent biological replication requires reviewing the numbered overlay below and editing a copied candidate TSV when the path does not follow the intended muscle/serosal boundary.


In [1]:
suppressPackageStartupMessages({
  library(SeuratObject)
  library(ggplot2)
})
options(stringsAsFactors = FALSE, repr.plot.width = 11, repr.plot.height = 10)
cat('R version:', R.version.string, '
')


R version: R version 4.6.1 (2026-06-24 ucrt) 


In [2]:
PROJECT_ROOT <- Sys.getenv(
  'COLON_PROJECT_ROOT',
  unset = if (.Platform$OS.type == 'windows') {
    'D:/Xiaonan/CODEX_projects/Yanan_Xenium'
  } else {
    '/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium'
  }
)
ANALYSIS_ROOT <- file.path(PROJECT_ROOT, 'colon_analysis')
PIPELINE_REPO <- file.path(ANALYSIS_ROOT, 'YNH_Xenium_Colon')
REGION_SPATIAL_RDS <- file.path(ANALYSIS_ROOT, 'HPC_return', '20260910_HPC_return', 'colon_downstream_outputs', 'full_notebook_qc_v2', '03_Region_6_Primary479', 'Region_6_spatial_passQC.rds')
TRACE_CONFIG <- file.path(PIPELINE_REPO, 'config', 'unrolling', 'Region_6_trace_control_points.tsv')
OUT_DIR <- file.path(ANALYSIS_ROOT, 'mayassi_unrolling_notebook', 'Region_6')
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
print(data.frame(name = c('PROJECT_ROOT','REGION_SPATIAL_RDS','TRACE_CONFIG','OUT_DIR'), path = c(PROJECT_ROOT,REGION_SPATIAL_RDS,TRACE_CONFIG,OUT_DIR)))


                name
1       PROJECT_ROOT
2 REGION_SPATIAL_RDS
3       TRACE_CONFIG
4            OUT_DIR
                                                                                                                                                                                     path
1                                                                                                                                               D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium
2 D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium/colon_analysis/HPC_return/20260910_HPC_return/colon_downstream_outputs/full_notebook_qc_v2/03_Region_6_Primary479/Region_6_spatial_passQC.rds
3                                                            D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium/colon_analysis/YNH_Xenium_Colon/config/unrolling/Region_6_trace_control_points.tsv
4                                                                                            D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium/colon_analysis/m

In [3]:
if (.Platform$OS.type == 'windows' && grepl('^[Cc]:', normalizePath(tempdir(), winslash = '/'))) stop('R temporary files are on C:. Set TMPDIR, TEMP, and TMP below colon_analysis/tmp.')
required_paths <- c(PIPELINE_REPO, REGION_SPATIAL_RDS, TRACE_CONFIG)
if (any(!file.exists(required_paths))) stop('Missing required path(s): ', paste(required_paths[!file.exists(required_paths)], collapse = '; '))
source(file.path(PIPELINE_REPO, 'R', 'source.R'))
cat('Environment and path checks: PASS
')


Environment and path checks: PASS


## Step 1 - Load the exact pass-QC population

No QC mask is recalculated here. The returned Seurat object is the starting population requested for unrolling. The notebook verifies that every retained cell carries `primary_include_revised == TRUE`.


In [4]:
region_spatial <- readRDS(REGION_SPATIAL_RDS)
spatial_meta <- region_spatial[[]]
required_metadata <- c('x_centroid','y_centroid','primary_include_revised','RefAll_subtype_predicted.id')
require_columns(spatial_meta, required_metadata, 'Region 6 pass-QC metadata')
reference_smooth <- as.character(spatial_meta$RefAll_subtype_predicted.id) == 'Smooth muscle'
input_audit <- data.frame(
  input_cells = nrow(spatial_meta),
  revised_primary_true = sum(spatial_meta$primary_include_revised %in% TRUE),
  revised_primary_not_true = sum(!(spatial_meta$primary_include_revised %in% TRUE)),
  reference_smooth_muscle_cells = sum(reference_smooth, na.rm = TRUE),
  missing_reference_labels = sum(is.na(spatial_meta$RefAll_subtype_predicted.id))
)
print(input_audit)
stopifnot(input_audit$revised_primary_not_true == 0L)


  input_cells revised_primary_true revised_primary_not_true
1      127935               127935                        0
  reference_smooth_muscle_cells missing_reference_labels
1                          8059                        0


## Step 2 - Load and validate the version-controlled manual coordinates

The TSV is the immutable input for exact replication. `point_order` defines the direction from outer free edge to inner endpoint. The comments identify the two deliberate inter-turn bridges and the inner hook. All 39 rows are printed so there is no hidden coordinate state.


In [5]:
control_points <- utils::read.delim(TRACE_CONFIG, check.names = FALSE)
control_audit <- validate_trace_control_points(control_points)
print(control_audit)
print(control_points, row.names = FALSE)


  n_control_points start_x start_y end_x end_y min_segment_um median_segment_um
1               39    4500     720  2825  2550       235.8495          708.9771
  max_segment_um
1       961.7692
 point_order    x    y                     comment
           1 4500  720             outer free edge
           2 4900  950                  outer turn
           3 5200 1450                  outer turn
           4 5330 2100                  outer turn
           5 5300 2900                  outer turn
           6 5100 3800                  outer turn
           7 4650 4650                  outer turn
           8 3900 5250                  outer turn
           9 3100 5510                  outer turn
          10 2300 5420                  outer turn
          11 1550 5150                  outer turn
          12  950 4750                  outer turn
          13  600 4200                  outer turn
          14  480 3500                  outer turn
          15  500 2850                  o

In [6]:
segment_audit <- data.frame(
  from_point = head(control_points$point_order, -1),
  to_point = tail(control_points$point_order, -1),
  segment_um = sqrt(diff(control_points$x)^2 + diff(control_points$y)^2),
  destination_comment = tail(control_points$comment, -1)
)
print(segment_audit[order(segment_audit$segment_um, decreasing = TRUE), ], row.names = FALSE)


 from_point to_point segment_um         destination_comment
          6        7   961.7692                  outer turn
          7        8   960.4686                  outer turn
          5        6   921.9544                  outer turn
          8        9   841.1896                  outer turn
         20       21   829.6987                 second turn
         26       27   816.3333                 second turn
          9       10   805.0466                  outer turn
          4        5   800.5623                  outer turn
         19       20   800.5623                 second turn
         10       11   797.1198                  outer turn
         25       26   793.9773                 second turn
         16       17   777.8175                 second turn
         23       24   751.0659                 second turn
         27       28   750.2666                 second turn
         24       25   743.3034                 second turn
         11       12   721.1103         

## Step 3 - Review every numbered point against the reference-transfer guide

Grey points are all pass-QC cells. Blue cells have `RefAll_subtype_predicted.id == "Smooth muscle"`. The green polyline is the proposed ordered boundary and red labels are the exact control-point numbers.

Review criteria: follow one continuous outer muscle/serosal band; do not jump to an adjacent coil merely because it is spatially close; inspect points 16 and 31 as explicit inter-turn bridges; and confirm that points 37-39 follow the inner hook.


In [7]:
plot_data <- data.frame(
  x = as.numeric(spatial_meta$x_centroid),
  y = as.numeric(spatial_meta$y_centroid),
  reference_smooth = reference_smooth
)
numbered_trace_plot <- ggplot(plot_data, aes(x, y)) +
  geom_point(color = 'grey82', size = 0.05, alpha = 0.18) +
  geom_point(data = plot_data[plot_data$reference_smooth %in% TRUE, ], color = '#2166AC', size = 0.12, alpha = 0.65) +
  geom_path(data = control_points, aes(x, y), inherit.aes = FALSE, color = '#00A651', linewidth = 0.65) +
  geom_point(data = control_points, aes(x, y), inherit.aes = FALSE, color = '#D73027', size = 1.2) +
  geom_text(data = control_points, aes(x, y, label = point_order), inherit.aes = FALSE, color = '#7F0000', size = 2.5, nudge_y = 55) +
  coord_equal() + theme_void() +
  labs(title = 'Region 6 manual trace review', subtitle = 'blue: Mayassi-reference Smooth muscle; green: ordered path; red numbers: editable control points')
ggsave(file.path(OUT_DIR, 'Region_6_numbered_trace_definition.png'), numbered_trace_plot, width = 10, height = 10, dpi = 220, bg = 'white')
numbered_trace_plot


![Numbered Region 6 manual trace](../../mayassi_unrolling_notebook/Region_6/Region_6_numbered_trace_definition.png)


## Step 4 - Save an auditable review copy

The repository TSV remains the canonical input. This step writes a dated-run-independent review copy and an audit table beside the downstream projection outputs. Editing should be performed on a copied candidate TSV, visually reviewed, and only then promoted to the repository configuration.


In [8]:
dense_trace <- add_trace_arc_length(interpolate_trace_control_points(control_points[, c('x','y')], spacing = 20))
trace_definition_audit <- data.frame(
  input_cells = nrow(spatial_meta),
  revised_primary_true = sum(spatial_meta$primary_include_revised %in% TRUE),
  reference_smooth_muscle_cells = sum(reference_smooth, na.rm = TRUE),
  control_points = nrow(control_points),
  dense_trace_points = nrow(dense_trace),
  dense_trace_arc_length_um = max(dense_trace$roll_arc_length),
  config_md5 = unname(tools::md5sum(TRACE_CONFIG))
)
utils::write.table(control_points, file.path(OUT_DIR, 'Region_6_trace_control_points.reviewed.tsv'), sep = '	', quote = FALSE, row.names = FALSE)
utils::write.table(trace_definition_audit, file.path(OUT_DIR, 'Region_6_trace_definition_audit.tsv'), sep = '	', quote = FALSE, row.names = FALSE)
print(trace_definition_audit)


  input_cells revised_primary_true reference_smooth_muscle_cells control_points
1      127935               127935                          8059             39
  dense_trace_points dense_trace_arc_length_um                       config_md5
1               1251                  24978.31 e7b703c64c4670e06ea9ead495dc7783


## Decision before B3

This notebook provides exact computational provenance, but it does not convert a manual biological annotation into an objective ground truth. If the numbered line crosses an unsupported tissue gap or follows the wrong coil, stop and revise a candidate trace before running B3. The current trace remains provisional until histology or expert review confirms it.


In [9]:
sessionInfo()


R version 4.6.1 (2026-06-24 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 22631)

Matrix products: default
  LAPACK version 3.12.1

locale:
[1] C
system code page: 65001

time zone: Asia/Shanghai
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] ggplot2_4.0.3      SeuratObject_5.4.0 sp_2.2-3           jsonlite_2.0.0    

loaded via a namespace (and not attached):
 [1] Matrix_1.7-5        gtable_0.3.6        future.apply_1.20.2
 [4] dplyr_1.2.1         compiler_4.6.1      tidyselect_1.2.1   
 [7] Rcpp_1.1.2          parallel_4.6.1      textshaping_1.0.5  
[10] systemfonts_1.3.2   globals_0.19.1      scales_1.4.0       
[13] lattice_0.22-9      R6_2.6.1            labeling_0.4.3     
[16] generics_0.1.4      dotCall64_1.2       future_1.75.0      
[19] tibble_3.3.1        pillar_1.11.1       RColorBrewer_1.1-3 
[22] rlang_1.3.0         S7_0.2.2            c